Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
#@title Imports (run this first)

import matplotlib.pyplot as plt
import numpy as np

# Erdős' minimum overlap problem

Let $C_5$ be the largest constant satisfying
$$ \sup_{x \in [-2,2]} \int_{-1}^1 f(t) g(x+t)\ dt\geq C_5$$
for all non-negative $f,g: [-1,1] \to [0,1]$ with $f+g=1$ on $[-1,1]$ and
$\int f(x)\ dx = 1$, where we extend $f,g$ by zero outside of $[-1,1]$. This
constant controls the asymptotics of the Minimum Overlap Problem described by
[Erdős (1955)](https://link.springer.com/article/10.1007/BF02760020). The bounds
$$0.379005 \leq C_5 \leq 0.380927$$
are known; the lower bound was obtained by
[White (2023)](https://www.impan.pl/en/publishing-house/journals-and-series/acta-arithmetica/online/115217/a-new-bound-for-erdos-minimum-overlap-problem)
via convex programming methods.

It is known (see [Haugland (2016)](https://arxiv.org/abs/1609.08000)) that
$C_5$ is equal to the infimum, over all step
functions $h$ on $[0, 2]$ with values in $[0, 1]$ and satisfying
$
\int_0^2 h(x)dx = 1
$
of
$$
\max_k \int h(x)(1 - h(x+k))dx
.$$ The upper bound to $C_5$ was obtained by using this result, in
[Haugland (2016)](https://arxiv.org/abs/1609.08000), via a step function
construction.

AlphaEvolve discovered an alternative step function construction that improves
the upper bound to $C_5 < 0.380924$.

In [ ]:
#@title Data

best_sequence = np.array([
    0.0, 0.0, 0.0, 0.0, 0.0,
    0.0, 3.60911302e-10, 3.62124044e-10, 4.02849974e-12,
    4.47352578e-12, 4.76914172e-12, 0.506074303,
    0.632046692, 0.679332798, 0.888193865,
    0.889214704, 0.678231235, 2.976636922840846e-07,
    0.0947643739, 0.0143926342, 0.423931858,
    0.598073612, 0.803909612, 0.683098916,
    0.314749384, 0.404059484, 0.858443734,
    0.796503042, 0.590433152, 0.41056218,
    0.270932695, 0.613384276, 0.709501647,
    0.580573615, 0.803538112, 0.715263878,
    0.822611331, 0.808433879, 0.683533985,
    0.645719012, 0.889417725, 0.943389845,
    0.841536959, 0.794505216, 0.941943428,
    0.962223227, 0.961270753, 0.992409079,
], dtype=np.float64)

In [ ]:
#@title Verification and plotting code

def plot_step_function(step_heights_input: list[float]):
  num_steps = len(step_heights_input)

  # Generate x values for plotting (need to plot steps properly).
  step_edges_plot = np.linspace(-0.25, 0.25, num_steps + 1)
  x_plot = np.array([])
  y_plot = np.array([])

  for i in range(num_steps):
    x_start = step_edges_plot[i]
    x_end = step_edges_plot[i + 1]
    x_step_vals = np.linspace(x_start, x_end, 100)  # Points within each step.
    y_step_vals = np.full_like(x_step_vals, step_heights_input[i])
    x_plot = np.concatenate((x_plot, x_step_vals))
    y_plot = np.concatenate((y_plot, y_step_vals))

  # Plot the step function.
  plt.figure(figsize=(8, 5))
  plt.plot(x_plot, y_plot)
  plt.xlabel("x")
  plt.ylabel("f(x)")
  plt.title(
      "Step function found by AlphaEvolve for Erdős' minimum overlap problem"
  )
  plt.xlim([-0.3, 0.3])
  plt.ylim([-1, max(step_heights_input) * 1.2])
  plt.grid(True)
  plt.step(
      step_edges_plot[:-1],
      step_heights_input,
      where='post',
      color='green',
      linewidth=2,
  )  # Overlay with plt.step for clarity.
  plt.show()


def verify_sequence(sequence: np.ndarray):
  """Raises an error if the sequence does not satisfy the constraints."""
  n = len(sequence)
  target_mirrored_sum = (2 * n - 1) / 2.0
  actual_mirrored_sum = 2 * np.sum(sequence) - sequence[-1]

  assert np.all((sequence >= 0) & (sequence <= 1)), "All values in the sequence must be between 0 and 1."
  assert np.isclose(
      actual_mirrored_sum, target_mirrored_sum, atol=1e-20
  ), f"The sum of values in the sequence must be exactly {target_mirrored_sum}. The sum is {actual_mirrored_sum}."
  print(f"The resulting step function has {2 * len(sequence) - 1} intervals and generates a valid step function for Erdős' minimum overlap problem.")


def compute_upper_bound(mirrored_sequence: np.ndarray) -> float:
  """Returns the upper bound given the mirrored sequence."""
  convolution_values = np.correlate(
      np.array(mirrored_sequence), 1 - np.array(mirrored_sequence), mode='full'
  )
  return np.max(convolution_values) / len(mirrored_sequence) * 2

In [ ]:
#@title Verification and plotting

verify_sequence(best_sequence)
mirrored_sequence = np.concatenate((best_sequence[:-1], best_sequence[::-1]))

print(
    f"The sequence provides the following upper bound on Erdős' minimum overlap problem: C5 <= {repr(float(compute_upper_bound(mirrored_sequence)))}."
)
plot_step_function(mirrored_sequence)